# ASG Airlines — Gold KPI Summary Table

**Stage:** Gold (reads `gold/fact_flights`, `gold/dim_airline`, `gold/dim_route`; writes `gold/gold_kpi_summary`)

## Why this table exists, and why it's deliberately disconnected

`gold_kpi_summary` is a single-row table of unfiltered, global totals (total flights, average duration, overnight/anomaly counts and percentages, distinct airline/route counts). It's built **separately from the star schema on purpose** — it has no relationship to `fact_flights` in the Power BI data model.

The reasoning: every other KPI in this project is computed as a live DAX measure against `fact_flights`, which means it correctly responds to slicers (e.g. filtering to one airline). `gold_kpi_summary` is the opposite by design — a fixed snapshot of the *true overall* numbers, useful for an "all-time baseline" reference that doesn't move when someone applies a filter elsewhere on the dashboard.

## Step 1 — Storage configuration & read Gold tables

In [1]:
import os
from datetime import datetime, timezone
import pandas as pd
from deltalake import write_deltalake, DeltaTable

STORAGE_ACCOUNT_NAME = "stasgairlines01"
CONTAINER_GOLD       = "gold"

STORAGE_KEY = os.environ.get("ADLS_STORAGE_KEY", "")
if not STORAGE_KEY:
    try:
        import subprocess
        res = subprocess.run(
            ["az", "storage", "account", "keys", "list",
             "--account-name", STORAGE_ACCOUNT_NAME,
             "--resource-group", "rg-asg-airlines",
             "--query", "[0].value", "-o", "tsv"],
            capture_output=True, text=True, check=True
        )
        STORAGE_KEY = res.stdout.strip()
    except Exception:
        raise RuntimeError("ADLS_STORAGE_KEY not set and could not fetch.")

so = {
    "azure_storage_account_name": STORAGE_ACCOUNT_NAME,
    "azure_storage_access_key": STORAGE_KEY,
}

fact_flights = DeltaTable(f"az://{CONTAINER_GOLD}/fact_flights", storage_options=so).to_pandas()
dim_airline  = DeltaTable(f"az://{CONTAINER_GOLD}/dim_airline", storage_options=so).to_pandas()
dim_route    = DeltaTable(f"az://{CONTAINER_GOLD}/dim_route", storage_options=so).to_pandas()

print(f"fact_flights: {len(fact_flights):,} rows | dim_airline: {len(dim_airline)} | dim_route: {len(dim_route)}")

fact_flights: 1,003 rows | dim_airline: 5 | dim_route: 30


## Step 2 — Calculate summary metrics

In [1]:
total_flights          = len(fact_flights)
avg_duration_minutes   = round(fact_flights["duration_minutes"].mean(), 1)
overnight_flight_count = fact_flights["is_overnight"].sum()
overnight_flight_pct   = round((overnight_flight_count / total_flights) * 100, 1) if total_flights > 0 else 0.0

anomaly_flight_count   = fact_flights["is_anomaly"].sum()
anomaly_flight_pct     = round((anomaly_flight_count / total_flights) * 100, 1) if total_flights > 0 else 0.0

distinct_airline_count = len(dim_airline)
distinct_route_count   = len(dim_route)

snapshot_generated_at  = datetime.now(timezone.utc)

print(f"Total flights: {total_flights}  Avg duration: {avg_duration_minutes} min")
print(f"Overnight: {overnight_flight_count} ({overnight_flight_pct}%)  Anomalies: {anomaly_flight_count} ({anomaly_flight_pct}%)")

Total flights: 1003  Avg duration: 163.2 min
Overnight: 122 (12.2%)  Anomalies: 1 (0.1%)


## Step 3 — Build the summary row and write it to Gold

In [1]:
summary_data = {
    "total_flights": [total_flights],
    "avg_duration_minutes": [avg_duration_minutes],
    "overnight_flight_count": [overnight_flight_count],
    "overnight_flight_pct": [overnight_flight_pct],
    "anomaly_flight_count": [anomaly_flight_count],
    "anomaly_flight_pct": [anomaly_flight_pct],
    "distinct_airline_count": [distinct_airline_count],
    "distinct_route_count": [distinct_route_count],
    "snapshot_generated_at": [snapshot_generated_at]
}

df_summary = pd.DataFrame(summary_data)

write_deltalake(f"az://{CONTAINER_GOLD}/gold_kpi_summary", df_summary, mode="overwrite", schema_mode="overwrite", storage_options=so)
print(f"Wrote gold_kpi_summary -> az://{CONTAINER_GOLD}/gold_kpi_summary")
df_summary

Wrote gold_kpi_summary -> az://gold/gold_kpi_summary


## Step 4 — Validation checks

In [1]:
print("=" * 80)
print("GOLD KPI SUMMARY VALIDATION REPORT")
print("=" * 80)

print("\n1. Single-row output dump:")
for col in df_summary.columns:
    print(f"   {col:<25} : {df_summary.iloc[0][col]}")

print("\n2. Sanity check — overnight + non-overnight should equal total")
non_overnight = total_flights - overnight_flight_count
check_sum = overnight_flight_count + non_overnight
print(f"   {overnight_flight_count} + {non_overnight} = {check_sum} (expected {total_flights})")
print("   -> PASS" if check_sum == total_flights else "   -> FAIL")

print("\n3. Sanity check — anomaly count within valid range")
print(f"   Anomalies: {anomaly_flight_count} / {total_flights} ({anomaly_flight_pct}%)")
print("   -> PASS" if anomaly_flight_count <= total_flights else "   -> FAIL")

print("\nGold KPI summary generation complete.")

GOLD KPI SUMMARY VALIDATION REPORT

1. Single-row output dump:
   total_flights             : 1003
   avg_duration_minutes      : 163.2
   overnight_flight_count    : 122
   overnight_flight_pct      : 12.2
   anomaly_flight_count      : 1
   anomaly_flight_pct        : 0.1
   distinct_airline_count    : 5
   distinct_route_count      : 30
   snapshot_generated_at     : 2026-09-11 12:04:33.128374+00:00

2. Sanity check — overnight + non-overnight should equal total
   122 + 881 = 1003 (expected 1003)
   -> PASS

3. Sanity check — anomaly count within valid range
   Anomalies: 1 / 1003 (0.1%)
   -> PASS

Gold KPI summary generation complete.
